In [0]:
import json

# ============================================================
# Source Onboarding Agent tools.
# The agent proposes; these tools check, test, ask the human and only then act.
# All network/storage work is delegated to _utils_bronze_framework.
# ============================================================


def _no_data(sample):
    return sample is None or sample.strip() in ("", "[]", "{}", "null")


def _dry_run_days(spec):
    """Smallest window that still produces at least one chunk (APIs with a delay need more days)."""
    incremental = spec["incremental"]
    offsets = [incremental.get("end_offset_days", 0)] + [
        ep.get("end_offset_days", incremental.get("end_offset_days", 0)) for ep in spec["endpoints"]
    ]
    return 1 + max(0, -min(offsets))


def propose_source_spec(spec_yaml):
    """Agent tool: parse -> validate -> dry run -> human review -> save YAML -> initial load."""

    # 1. Parse YAML
    try:
        spec = load_source_spec(spec_yaml)
    except ValueError as e:
        print(f"🛡️ Spec rejected (YAML): {str(e)[:150]}")
        return json.dumps({
            "status": "rejected", "stage": "yaml", "errors": [str(e)],
            "instruction": "Fix the YAML syntax. Quote values that start with { or %, or that contain ': '.",
        })

    # 2. Validate rules + name not already taken
    errors = validate_source_spec(spec)
    if spec.get("source_name") in list_source_specs():
        errors.append(f"source '{spec['source_name']}' already exists; choose another source_name or ask the user")
    if errors:
        print(f"🛡️ Spec rejected by validation: {errors}")
        return json.dumps({
            "status": "rejected", "stage": "validation", "errors": errors,
            "instruction": "Change the spec according to the errors. Do not resend the same spec.",
        })

    # 3. Dry run: the API must answer AND return data
    try:
        summary = run_source_spec(spec, days_back_override=_dry_run_days(spec), dry_run=True)
    except Exception as e:
        return json.dumps({"status": "error", "stage": "dry_run", "errors": [str(e)[:500]]})

    problems = [f"{r['endpoint_name']} {r['chunk_start']}->{r['chunk_end']}: {r['error_message']}"
                for r in summary["results"] if r["status"] == "error"]
    problems += [f"{r['endpoint_name']}: the API answered but returned no data; check URL, params and dates"
                 for r in summary["results"] if r["status"] != "error" and _no_data(r["sample"])]
    if not summary["results"]:
        problems.append("dry run produced no chunks; check days_back and end_offset_days")
    if problems:
        print(f"🧪 Dry run failed, the agent will fix it: {problems}")
        return json.dumps({
            "status": "error", "stage": "dry_run", "errors": problems,
            "instruction": "Probe the endpoint again if needed, fix the spec and propose it again.",
        })

    # 4. Human review: only for specs that already work
    print("\n===== THE AGENT PROPOSES A NEW SOURCE =====\n")
    print(spec_to_yaml(spec))
    shown = set()
    for r in summary["results"]:
        if r["endpoint_name"] not in shown:
            shown.add(r["endpoint_name"])
            print(f"🧪 Dry run sample · {r['endpoint_name']} ({r['chunk_start']}->{r['chunk_end']}):")
            print(r["sample"][:400], "\n")

    answer = input("Approve this source? (y = approve / n = reject / or type your feedback): ").strip()

    if answer.lower() in ("n", "no", ""):
        return json.dumps({"status": "rejected_by_human",
                           "message": "The user did not approve the new source. Tell the user."})

    if answer.lower() not in ("y", "yes"):
        print(f"📝 Feedback sent to the agent: {answer}")
        return json.dumps({
            "status": "changes_requested",
            "feedback": answer,
            "instruction": "The human reviewer requested changes. Apply the feedback exactly, "
                           "probe again if needed, and propose the revised spec.",
        })

    # 5. Save the spec (versioned in Git) and run the initial load
    path = save_source_spec_file(spec)
    print(f"\n💾 Spec saved: {path}\n")
    load = run_source_spec(spec)

    print(f"\n✅ Source {spec['source_name']} onboarded · {load['chunks_ok']} ok / {load['chunks_error']} errors")
    return json.dumps({
        "status": "created",
        "source_name": spec["source_name"],
        "spec_file": path,
        "chunks_ok": load["chunks_ok"],
        "chunks_error": load["chunks_error"],
        "bronze_tables": list(load["bronze_tables"].keys()),
    })

In [0]:
import os
import mlflow
from datetime import date

max_agent_steps = 20
max_spec_proposals = 5
max_tool_result_chars = 6000
example_sources = ["ree", "open_meteo_archive"]   # specs shown to the agent as examples

_llm_client = None


def get_llm_client():
    """Creates the LLM client only when the agent runs (tests of the framework do not need LLM packages)."""
    global _llm_client
    if _llm_client is None:
        from databricks.sdk import WorkspaceClient
        mlflow.openai.autolog()
        _llm_client = WorkspaceClient().serving_endpoints.get_open_ai_client()   # OpenAI-compatible client
    return _llm_client


# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------
def extract_text(content):
    """Text from a model response (works with reasoning models that return blocks)."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        block_type = block.get("type") if isinstance(block, dict) else getattr(block, "type", None)
        text = block.get("text") if isinstance(block, dict) else getattr(block, "text", None)
        if block_type == "text" and text:
            parts.append(text)
    return "\n".join(parts)


def list_existing_sources():
    sources = []
    for name in list_source_specs():
        spec = load_source_spec_file(name)
        sources.append({
            "source_name": name,
            "description": spec.get("description"),
            "endpoints": [{"name": ep.get("name"), "url": ep.get("url"), "params": ep.get("params", {}),
                           "locations": ep.get("locations")}
                          for ep in spec.get("endpoints", [])],
        })
    return json.dumps(sources, ensure_ascii=False)


def _example_specs_text():
    texts = []
    for name in example_sources:
        if name in list_source_specs():
            with open(os.path.join(find_source_specs_dir(), f"{name}.yml"), encoding="utf-8") as f:
                texts.append(f"# --- example: {name}.yml ---\n{f.read()}")
    return "\n".join(texts)


def _reference_files_text():
    """Reference files the agent can use in a locations block (read live from reference_data/)."""
    lines = []
    folder = find_reference_data_dir()
    for file_name in sorted(os.listdir(folder)):
        if not file_name.endswith(".csv"):
            continue
        rows = read_reference_csv(file_name)
        if rows and {"latitude", "longitude"} <= set(rows[0]):
            lines.append(f"- {file_name}: {len(rows)} rows, columns {list(rows[0].keys())}")
    return "\n".join(lines) or "- (none)"


# ------------------------------------------------------------
# 2. Deterministic onboarding report (facts come from the saved spec, not from the LLM)
# ------------------------------------------------------------
def _describe_load_strategy(spec):
    incremental = spec["incremental"]
    if incremental["type"] == "snapshot":
        return "snapshot: full download on every run (one file per run date, previous snapshots are kept)"
    offset = incremental.get("end_offset_days", 0)
    if offset == 0:
        until = "today"
    elif offset > 0:
        until = f"today + {offset} day(s) (future dates, e.g. forecasts)"
    else:
        until = f"today - {-offset} day(s) (the source publishes with a delay)"
    return (f"date_range: last {incremental['days_back']} day(s) up to {until}, "
            f"in calendar-aligned chunks of {incremental['chunk_days']} day(s); "
            f"re-runs overwrite the same chunks (latest version wins)")


def build_onboarding_report(outcome):
    spec = load_source_spec_file(outcome["source_name"])
    lines = [
        f"### ✅ Source onboarded: `{spec['source_name']}`",
        f"- **Description:** {spec['description']}",
        f"- **Spec file:** `{outcome['spec_file']}` (commit it in Git to complete the review)",
        f"- **Load strategy:** {_describe_load_strategy(spec)}",
        "- **Endpoints:**",
    ]
    for ep in spec["endpoints"]:
        line = f"  - `{ep['name']}`: {ep['url']} ({ep['response_format']})"
        if ep.get("locations"):
            n_locations = len(read_reference_csv(ep["locations"]["reference"]))
            line += f" · {n_locations} locations from `reference_data/{ep['locations']['reference']}`"
        lines.append(line)
    lines += [
        f"- **Initial load:** {outcome['chunks_ok']} chunk(s) ok, {outcome['chunks_error']} error(s)",
        "- **Bronze tables:** " + ", ".join(f"`{t}`" for t in outcome["bronze_tables"]),
        "- **Schedule:** not scheduled yet; data is refreshed only when the ingestion runs",
    ]
    return "\n".join(lines)


# ------------------------------------------------------------
# 3. Agent instructions
# ------------------------------------------------------------
onboarding_prompt_template = """You are the Source Onboarding Agent of a governed data platform (Databricks, medallion architecture).
Your job: turn a user's request for a new data source into a SOURCE SPEC (YAML). You never write code.
A deterministic framework validates your spec, runs a dry run, asks a human for review and only then ingests the data into bronze.
Today is __TODAY__.
Always write in English, whatever language the user writes in.

PROCESS
1. Call list_existing_sources first. Do not onboard a source that already exists.
   Reuse what existing sources already do (variable names, locations block, timezone) whenever the user asks
   for something similar or "the same as" an existing source.
2. Identify the public API that provides the requested data, from your own knowledge. You cannot search the web.
   If you do not know a suitable API, ask the user for the API documentation URL instead of guessing.
3. Verify with probe_url before proposing: use real parameters and a SMALL date range of 1-3 days
   close to today (__TODAY__). Never probe with dates from previous years unless the user asks for them.
   For APIs with locations, probe with 1-2 coordinates only; the full list is added by the framework.
   Read the structure: check the response format, the date format the API expects and that data is actually returned.
   If a probe fails, read the http_status and error, adjust the parameters and try again (at most 3 attempts).
4. Write the spec and call propose_source_spec with the full YAML.
   - status rejected or error: read the errors, probe again if needed, fix the spec and propose again.
   - status changes_requested: the human reviewer wrote feedback. Apply it exactly and propose the revised spec.
   Never resend the same spec. You have at most __MAX_PROPOSALS__ proposals.
5. When the tool returns status created or rejected_by_human, stop and write the final answer.

SPEC FORMAT (YAML)
source_name: snake_case, 3-41 chars, unique (e.g. open_meteo_forecast)
source_type: http_api                      # the only supported type
description: "quoted text: what the data is, provider, coverage"
auth:
  type: none                               # only public APIs without authentication
incremental:
  type: date_range                         # date_range (data by date) or snapshot (full download each run)
  days_back: 90                            # date_range only: integer 1-730, history to load
  chunk_days: 7                            # date_range only: integer 1-31, days per API call; keep responses small
  date_format: "%Y-%m-%d"                  # date_range only: one of "%Y-%m-%d", "%Y%m%d", "%d/%m/%Y"
  end_offset_days: 0                       # date_range only: integer -10..2; 0 = up to today,
                                           #   negative if the API publishes with a delay, positive for forecasts
endpoints:                                 # one or more
  - name: snake_case
    url: https://...                       # https only, public host, NO query string in the url
    method: GET                            # GET only
    response_format: json                  # json or csv
    csv_delimiter: ";"                     # optional, csv only: one of , ; | and tab
    params:                                # query parameters: strings or numbers
      some_param: value
      start_param: "{start}"               # date_range: the chunk start date, formatted with date_format
      end_param: "{end}"                   # date_range: the chunk end date
    locations:                             # optional: for APIs that accept lists of coordinates
      reference: locations_es.csv          # a file from the AVAILABLE REFERENCE FILES below
      key_column: province_code            # column that identifies each location
      latitude_param: latitude             # API parameter that receives the comma-separated latitudes
      longitude_param: longitude           # API parameter that receives the comma-separated longitudes
Rules:
- date_range endpoints must use BOTH {start} and {end}; snapshot endpoints must not use any placeholder.
- {start} and {end} are the only placeholders. They can be combined with text, e.g. "{start}T00:00".
- Chunks are aligned to the calendar and re-runs OVERWRITE them: only the latest version of each date is kept.
- locations: the framework fills latitude_param and longitude_param from the reference file and stores the
  location order in bronze. Do NOT put those parameters in params, and never write coordinate lists by hand
  when a reference file covers the requested places (e.g. "all Spanish provinces").
- YAML quoting: quote any value that starts with { or %, that contains ': ', and lists of numbers like "40.4,37.3".

AVAILABLE REFERENCE FILES (reference_data/)
__REFERENCE_FILES__

EXAMPLES (approved specs already in production)
__EXAMPLE_SPECS__

LIMITS OF THE FRAMEWORK
If the source needs something the spec cannot express (authentication/API key, POST, pagination,
a year or other value in the URL that changes over time, XML/Excel/ZIP), do not force a hack that loses data.
Either propose a reasonable workaround AND explain its limitation, or explain which framework extension is needed.

FINAL ANSWER (in English)
- If the source was CREATED: the framework already shows the user a factual summary (source, endpoints,
  load strategy, tables, schedule). Do NOT repeat it. Write only observations and limitations,
  at most 5 short bullet points, checking:
  1. Does overwriting chunks lose data the user may want to keep (e.g. past forecasts replaced by newer ones)?
  2. Is the naming consistent with existing sources (variable names, units, locations, timezone)?
  3. History or coverage limits of the API.
  4. Anything else the data owner should decide.
  Only state facts that follow from the spec or from what you probed; if you are not sure, say so.
- If the source was NOT created: explain why, ALWAYS including the exact technical reason
  (tool, URL, http_status and error message) so the user can investigate."""


def build_onboarding_prompt():
    return (onboarding_prompt_template
            .replace("__TODAY__", f"{date.today()} ({date.today().strftime('%A')})")
            .replace("__MAX_PROPOSALS__", str(max_spec_proposals))
            .replace("__REFERENCE_FILES__", _reference_files_text())
            .replace("__EXAMPLE_SPECS__", _example_specs_text()))


# ------------------------------------------------------------
# 4. Tools exposed to the LLM
# ------------------------------------------------------------
onboarding_tools = [
    {"type": "function", "function": {
        "name": "list_existing_sources",
        "description": "Lists the sources already onboarded (name, description, endpoints with URL, params and locations). "
                       "Call it first to avoid duplicates and to reuse existing naming and locations.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "probe_url",
        "description": (
            "Safe exploratory GET to a public https API. Returns HTTP status, detected format, "
            "the JSON structure with example values (or a CSV preview) and a text sample. Nothing is stored. "
            "Put query parameters in params, never in the url. Use small date ranges close to today."
        ),
        "parameters": {"type": "object", "properties": {
            "url": {"type": "string", "description": "https URL without query string"},
            "params": {"type": "object", "description": "Query parameters as key/value pairs (strings or numbers)"},
        }, "required": ["url"]},
    }},
    {"type": "function", "function": {
        "name": "propose_source_spec",
        "description": (
            "Submits a complete source spec as YAML text. The framework validates it, runs a dry run "
            "and asks the human reviewer, who can approve, reject or request changes. "
            "If approved, it saves the spec and runs the initial load. Returns errors or feedback to act on."
        ),
        "parameters": {"type": "object", "properties": {
            "spec_yaml": {"type": "string", "description": "The full YAML spec"},
        }, "required": ["spec_yaml"]},
    }},
]


@mlflow.trace(span_type="TOOL")
def execute_onboarding_tool(tool_name, arguments, state):
    try:
        if tool_name == "list_existing_sources":
            result = list_existing_sources()
        elif tool_name == "probe_url":
            result = json.dumps(probe_url(arguments.get("url", ""), arguments.get("params")),
                                ensure_ascii=False, default=str)
        elif tool_name == "propose_source_spec":
            if state["proposals"] >= max_spec_proposals:
                state["finished"] = True
                result = json.dumps({"status": "rejected",
                                     "errors": [f"maximum of {max_spec_proposals} proposals reached"],
                                     "instruction": "Stop and explain to the user what failed."})
            else:
                state["proposals"] += 1
                result = propose_source_spec(arguments.get("spec_yaml", ""))
                outcome = json.loads(result)
                if outcome.get("status") == "created":
                    state["created"] = outcome
                if outcome.get("status") in ("created", "rejected_by_human"):
                    state["finished"] = True
        else:
            result = json.dumps({"error": f"Unknown tool {tool_name}"})
    except Exception as e:
        result = json.dumps({"error": f"{type(e).__name__}: {str(e)[:500]}"})
    return result[:max_tool_result_chars]


def _describe_call(tool_name, arguments):
    if tool_name == "probe_url":
        return f"{arguments.get('url')} {arguments.get('params') or ''}"
    if tool_name == "propose_source_spec":
        return f"(YAML, {len(str(arguments.get('spec_yaml', '')).splitlines())} lines)"
    return ""


def _describe_result(result):
    """One-line summary of a tool result, so the human can follow what the agent sees."""
    try:
        data = json.loads(result)
    except ValueError:
        return "→ (long result)"
    if isinstance(data, list):
        return f"→ {len(data)} item(s)"
    parts = [str(data[k]) for k in ("status", "stage", "http_status", "detected_format") if data.get(k) is not None]
    problem = data.get("errors") or data.get("error") or data.get("feedback") or data.get("warning") or data.get("hint")
    if problem:
        parts.append(str(problem)[:200])
    return "→ " + " · ".join(parts)


# ------------------------------------------------------------
# 5. The agent loop
# ------------------------------------------------------------
@mlflow.trace(span_type="AGENT")
def run_onboarding_agent(request, max_steps=max_agent_steps):
    llm_client = get_llm_client()
    reset_probe_budget()
    state = {"proposals": 0, "finished": False, "created": None}
    messages = [{"role": "system", "content": build_onboarding_prompt()},
                {"role": "user", "content": request}]
    previous_calls = set()

    for step in range(max_steps):
        # Once the human approved/rejected, the agent can only answer
        choice = "none" if state["finished"] else "auto"
        response = llm_client.chat.completions.create(
            model=llm_endpoint, messages=messages, tools=onboarding_tools, tool_choice=choice,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            answer = extract_text(message.content)
            if state["created"]:
                return build_onboarding_report(state["created"]) + "\n\n#### Agent observations\n\n" + answer
            return answer

        messages.append({
            "role": "assistant",
            "content": extract_text(message.content),
            "tool_calls": [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                           for tc in message.tool_calls],
        })

        for tc in message.tool_calls:
            try:
                arguments = json.loads(tc.function.arguments or "{}")
            except ValueError:
                arguments = None

            if arguments is None:
                result = json.dumps({"error": "Arguments are not valid JSON. Call the tool again with valid JSON."})
            else:
                call_key = (tc.function.name, json.dumps(arguments, sort_keys=True))
                if call_key in previous_calls:
                    print(f"[step {step + 1}] ⚠️ repeated call: {tc.function.name}")
                    result = json.dumps({"warning": "You already made this exact call; its result is earlier "
                                                    "in the conversation. Change something or write the final answer."})
                else:
                    previous_calls.add(call_key)
                    print(f"[step {step + 1}] {tc.function.name} {_describe_call(tc.function.name, arguments)}")
                    result = execute_onboarding_tool(tc.function.name, arguments, state)
                    print(f"           {_describe_result(result)}")

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "Reached the maximum number of steps without a final answer."